## [v3] 변경 이력

**문제**: c6(test) 후반부에서 진동(Vib_X/Y/Z 계열) 피처가 train(c1,c4) 기준
표준편차 15 이상까지 튀는 현상 발견. c6이 train보다 더 심하게 마모되는
cutter라서(EDA에서 확인: c1 165.2 / c4 203.1 / c6 215.9) 실제로 벌어지는
현상이지 데이터 오류는 아님. 하지만 모델이 이런 극단적인 입력을 만나면
예측이 0으로 무너지는 문제가 발생함 (대시보드에서 확인됨).

**수정**: 정규화(StandardScaler) 직후, 값을 ±5 표준편차로 clip.
train/val에는 이 정도로 튀는 값이 거의 없어 영향 미미하고, test(c6) 후반부에만
실질적으로 적용됨. 몇 개가 잘렸는지 확인하는 로그도 추가함.

**주의**: 이 노트북은 Colab에서 쓰신 최신 버전(Google Drive 경로,
`cutter_train.npy`/`cutter_val.npy` 저장 로직 추가된 버전)과 셀 구조가
다를 수 있습니다. 이 파일은 셀 15~17(분할+정규화+클리핑) 로직만 참고해서,
실제 사용하시는 노트북에 이 클리핑 블록만 옮겨 붙이시는 걸 권장합니다.

## [v2] 변경 이력

**문제**: 기존 버전은 c1/c4를 cut 번호 기준으로 통째로 시간순 8:2 분할했더니,
RUL=0 구간(수명 종료 이후)이 두 cutter 모두 뒤쪽 20%(=val)에 몰려서
**train에는 RUL=0 샘플이 단 하나도 없었음** (`y_train range: 18~257`).

**수정**: 셀 15에서 "RUL>0 구간"과 "RUL=0 구간"을 각각 따로 시간순 8:2로 나누도록 변경.
두 구간 모두 train/val에 비율대로 섞이게 되어, train에도 RUL=0 샘플이 포함됨.
여전히 랜덤 셔플이 아닌 시간순 분할이라 data leakage 우려는 없음.

test(c6) 및 정규화 로직(StandardScaler를 train에만 fit)의 기본 구조는 그대로 유지.

In [2]:
# ===== 1. 라이브러리 =====
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import skew, kurtosis
import pywt
from sklearn.preprocessing import StandardScaler

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ===== 2. 경로 및 컬럼 설정 =====
DATA_DIR = Path("archive")
RAW_DIR = DATA_DIR
CUTTERS = ["c1", "c4", "c6"]
COLUMN_NAMES = ["Force_X", "Force_Y", "Force_Z", "Vib_X", "Vib_Y", "Vib_Z", "AE_RMS"]

# ===== 3. 파일 목록 함수 =====
def list_cut_files(cutter_dir, cutter_name):
    cut_subdir = cutter_dir / cutter_name
    return sorted(cut_subdir.glob(f"c_{cutter_name[1:]}_*.csv"))

# ===== 4. wear 라벨 로드 함수 =====
def load_wear_labels(cutter_dir):
    cutter_name = cutter_dir.name
    wear_path = cutter_dir / f"{cutter_name}_wear.csv"
    wear_df = pd.read_csv(wear_path)
    if "cut" not in wear_df.columns:
        wear_df.insert(0, "cut", range(1, len(wear_df) + 1))
    flute_cols = ["flute_1", "flute_2", "flute_3"]
    wear_df["mean_wear"] = wear_df[flute_cols].mean(axis=1)
    wear_df["max_wear"] = wear_df[flute_cols].max(axis=1)
    return wear_df

# ===== 5. threshold=150 기반 RUL 계산 함수 =====
def compute_rul(wear_df, theta=150):
    wear_df = wear_df.copy()
    exceed = wear_df[wear_df["mean_wear"] >= theta]
    if len(exceed) > 0:
        t_fail = exceed["cut"].iloc[0]
    else:
        t_fail = wear_df["cut"].iloc[-1]
    wear_df["RUL"] = (t_fail - wear_df["cut"]).clip(lower=0)
    wear_df["t_fail"] = t_fail
    return wear_df

# ===== 6. 원시 신호 로드 =====
def load_signal(filepath):
    return pd.read_csv(filepath, header=None, names=COLUMN_NAMES)

# ===== 7. wavelet 에너지 계산 (db4, level 3) =====
def calculate_wavelet_energy(data, downsample=10):
    data_sub = data[::downsample]
    coeffs = pywt.wavedec(data_sub, 'db4', level=3)
    return [np.sum(np.square(c)) for c in coeffs]

# ===== 8. cut 하나에서 피처 추출 (61개: 49 시간영역 + Force 3채널 x wavelet 4레벨) =====
def extract_features_from_signal(df):
    features = {}
    for col in COLUMN_NAMES:
        x = df[col].values
        features[f"{col}_mean"] = x.mean()
        features[f"{col}_std"] = x.std()
        features[f"{col}_rms"] = np.sqrt(np.mean(x ** 2))
        features[f"{col}_max"] = x.max()
        features[f"{col}_p2p"] = x.max() - x.min()
        features[f"{col}_skew"] = skew(x)
        features[f"{col}_kurtosis"] = kurtosis(x)

        # Force 계열에만 wavelet 에너지 추가 (AE_RMS 파생 피처는 제외)
        if col.startswith("Force"):
            wave_energies = calculate_wavelet_energy(x)
            for i, e in enumerate(wave_energies):
                features[f"{col}_wave_energy_L{i}"] = e

    return features

# ===== 9. cutter 전체 피처 테이블 생성 =====
def build_feature_table(cutter):
    cutter_dir = RAW_DIR / cutter
    cut_files = list_cut_files(cutter_dir, cutter)
    rows = []
    for f in tqdm(cut_files, desc=f"{cutter} 피처 추출"):
        df = load_signal(f)
        feats = extract_features_from_signal(df)
        feats["cut"] = int(f.stem.split("_")[-1])
        rows.append(feats)
    feature_df = pd.DataFrame(rows).sort_values("cut").reset_index(drop=True)
    merged = feature_df.merge(wear_tables[cutter], on="cut", how="left")
    return merged

# ===== 10. 폴더 구조 확인 =====
for cutter in CUTTERS:
    cutter_dir = RAW_DIR / cutter
    if not cutter_dir.exists():
        print(f"[없음] {cutter_dir}")
        continue
    cut_files = list_cut_files(cutter_dir, cutter)
    print(f"{cutter}: cut 파일 {len(cut_files)}개 발견")

# ===== 11. wear 라벨 로드 + RUL 라벨링 (θ=150) =====
wear_tables = {}
for cutter in CUTTERS:
    cutter_dir = RAW_DIR / cutter
    if not cutter_dir.exists():
        continue
    wear_df = load_wear_labels(cutter_dir)
    wear_df = compute_rul(wear_df, theta=150)
    wear_tables[cutter] = wear_df
    t_fail = wear_df["t_fail"].iloc[0]
    pct = t_fail / len(wear_df) * 100
    print(f"{cutter}: t_fail=cut {t_fail} ({pct:.1f}%), RUL=0 개수={(wear_df['RUL']==0).sum()}")

# ===== 12. c1, c4, c6 피처 테이블 생성 (61개 피처) =====
feature_table_c1 = build_feature_table("c1")
feature_table_c4 = build_feature_table("c4")
feature_table_c6 = build_feature_table("c6")

feat_cols = [c for c in feature_table_c1.columns if any(c.startswith(s) for s in COLUMN_NAMES)]
print(f"\n최종 피처 개수: {len(feat_cols)}")
print(feature_table_c1.shape, feature_table_c4.shape, feature_table_c6.shape)

c1: cut 파일 315개 발견
c4: cut 파일 315개 발견
c6: cut 파일 315개 발견
c1: t_fail=cut 287 (91.1%), RUL=0 개수=29
c4: t_fail=cut 270 (85.7%), RUL=0 개수=46
c6: t_fail=cut 228 (72.4%), RUL=0 개수=88


c6 피처 추출: 100%|██████████████████████████████████████████████████████████████████| 315/315 [03:08<00:00,  1.67it/s]


최종 피처 개수: 61
(315, 69) (315, 69) (315, 69)


In [ ]:
# ===== 13. 슬라이딩 윈도우 구성 함수 (cut-level 피처 -> 시퀀스) =====
def build_sliding_windows(df, feat_cols, N):
    df = df.sort_values("cut").reset_index(drop=True)
    X, y, cuts = [], [], []
    for t in range(N - 1, len(df)):
        window = df.iloc[t - N + 1 : t + 1][feat_cols].values
        X.append(window)
        y.append(df.iloc[t]["RUL"])
        cuts.append(df.iloc[t]["cut"])
    return np.array(X), np.array(y), np.array(cuts)

# ===== 14. N=30으로 슬라이딩 윈도우 생성 (cutter별 독립) =====
N = 30
X_c1, y_c1, cuts_c1 = build_sliding_windows(feature_table_c1, feat_cols, N)
X_c4, y_c4, cuts_c4 = build_sliding_windows(feature_table_c4, feat_cols, N)
X_c6, y_c6, cuts_c6 = build_sliding_windows(feature_table_c6, feat_cols, N)

print(f"c1 윈도우: {X_c1.shape}, c4 윈도우: {X_c4.shape}, c6 윈도우: {X_c6.shape}")

# ===== 15. Train / Val / Test 분할 (cutter 단위, "구간별" 시간순 홀드아웃) =====
# [v2 변경] 기존에는 c1/c4 전체를 시간순으로 8:2 했더니, RUL=0 구간이
# 전부 뒤쪽(=val)에 몰려서 train에 RUL=0 샘플이 아예 없었음
# (y_train range가 18~257로, 0이 빠져있던 문제).
#
# 수정: "RUL>0(생존 구간)"과 "RUL=0(이미 EOL 지난 구간)"을 각각 따로
# 시간순 8:2로 나눔. 두 세그먼트 모두 train/val에 골고루 포함되도록 함.
# - 여전히 랜덤 셔플이 아니라 "시간순" 분할이라 leakage 문제는 그대로 방지됨
#   (같은 세그먼트 안에서만 앞 80%/뒤 20%로 자르는 것)
# - test(c6)는 그대로 전체 유지 (변경 없음)
def split_train_val_by_segment(X, y, cuts, val_ratio=0.2):
    """RUL>0 구간과 RUL=0 구간을 각각 시간순 (1-val_ratio):val_ratio로 나눠서,
    두 구간 모두 train/val에 포함되게 함. 랜덤 셔플 아님 - 시간순 그대로 유지."""
    alive_mask = y > 0
    dead_mask = ~alive_mask

    def _split_one_segment(mask):
        idx = np.where(mask)[0]  # cuts 순서대로 정렬된 인덱스
        cut_point = int(len(idx) * (1 - val_ratio))
        return idx[:cut_point], idx[cut_point:]

    alive_train_idx, alive_val_idx = _split_one_segment(alive_mask)
    dead_train_idx, dead_val_idx = _split_one_segment(dead_mask)

    train_idx = np.sort(np.concatenate([alive_train_idx, dead_train_idx]))
    val_idx = np.sort(np.concatenate([alive_val_idx, dead_val_idx]))
    return train_idx, val_idx


VAL_RATIO = 0.2
c1_train_idx, c1_val_idx = split_train_val_by_segment(X_c1, y_c1, cuts_c1, VAL_RATIO)
c4_train_idx, c4_val_idx = split_train_val_by_segment(X_c4, y_c4, cuts_c4, VAL_RATIO)

X_train = np.concatenate([X_c1[c1_train_idx], X_c4[c4_train_idx]])
y_train = np.concatenate([y_c1[c1_train_idx], y_c4[c4_train_idx]])
X_val = np.concatenate([X_c1[c1_val_idx], X_c4[c4_val_idx]])
y_val = np.concatenate([y_c1[c1_val_idx], y_c4[c4_val_idx]])
X_test, y_test = X_c6, y_c6

# 이후 정규화(scaler.fit) 단계에서 동일한 train 컷만 골라 쓰기 위해 cut 번호 저장
train_cuts_c1 = cuts_c1[c1_train_idx]
train_cuts_c4 = cuts_c4[c4_train_idx]

print(f"\nX_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"y_train range: {y_train.min():.0f} ~ {y_train.max():.0f}")
print(f"y_val range:   {y_val.min():.0f} ~ {y_val.max():.0f}")
print(f"y_test range:  {y_test.min():.0f} ~ {y_test.max():.0f}")
print(f"[v2 확인] y_train 중 RUL=0 개수: {(y_train==0).sum()} (개선 전에는 0개였음)")
print(f"[v2 확인] y_val 중 RUL=0 개수:   {(y_val==0).sum()}")

# ===== 16. 정규화 (StandardScaler, train에서만 fit) =====
# [v2 변경] train_rows를 "cut <= split" 기준이 아니라, 위에서 새로 정한
# train_cuts_c1 / train_cuts_c4 (RUL=0 포함) 기준으로 선택하도록 수정
train_rows = pd.concat([
    feature_table_c1[feature_table_c1["cut"].isin(train_cuts_c1)],
    feature_table_c4[feature_table_c4["cut"].isin(train_cuts_c4)],
])
scaler = StandardScaler()
scaler.fit(train_rows[feat_cols])

def scale_windows(X, scaler):
    batch, n, d = X.shape
    return scaler.transform(X.reshape(-1, d)).reshape(batch, n, d)

X_train = scale_windows(X_train, scaler)
X_val = scale_windows(X_val, scaler)
X_test = scale_windows(X_test, scaler)

# ===== [v3 추가] 정규화 후 극단값 클리핑 =====
# c6 후반부(마모 심한 구간)에서 진동(Vib) 계열 피처가 train 분포 기준으로
# 표준편차 15 이상까지 튀는 게 확인됨 (c6이 train보다 더 심하게 마모되는
# cutter라 실제로 벌어지는 현상 - 데이터 오류 아님). 모델이 이렇게 낯선
# 크기의 입력을 만나면 예측이 무너지는(0으로 쏠리는) 걸 막기 위해,
# 정규화된 값을 ±CLIP_SIGMA 표준편차로 잘라냄.
# train/val에는 이 정도로 튀는 값이 거의 없어서 사실상 영향 없고,
# test(c6) 후반부에만 실질적으로 적용됨.
CLIP_SIGMA = 5.0
n_clipped_train = (np.abs(X_train) > CLIP_SIGMA).sum()
n_clipped_val = (np.abs(X_val) > CLIP_SIGMA).sum()
n_clipped_test = (np.abs(X_test) > CLIP_SIGMA).sum()

X_train = np.clip(X_train, -CLIP_SIGMA, CLIP_SIGMA)
X_val = np.clip(X_val, -CLIP_SIGMA, CLIP_SIGMA)
X_test = np.clip(X_test, -CLIP_SIGMA, CLIP_SIGMA)

print(f"\n[v3 확인] \u00b1{CLIP_SIGMA} 밖이라 잘려나간 값 개수")
print(f"  X_train: {n_clipped_train} / {X_train.size} ({n_clipped_train / X_train.size * 100:.3f}%)")
print(f"  X_val:   {n_clipped_val} / {X_val.size} ({n_clipped_val / X_val.size * 100:.3f}%)")
print(f"  X_test:  {n_clipped_test} / {X_test.size} ({n_clipped_test / X_test.size * 100:.3f}%)  <- 대부분 여기서 나올 것")

print(f"\n정규화 완료")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

# ===== 17. 정규화 검증 =====
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_val_flat = X_val.reshape(-1, X_val.shape[-1])
X_test_flat = X_test.reshape(-1, X_test.shape[-1])

print("\nX_train 검증 (평균 0, std 1에 가까워야 함)")
print(f"  mean 절대값 최대: {np.abs(X_train_flat.mean(axis=0)).max():.6f}")
print(f"  std 범위: {X_train_flat.std(axis=0).min():.4f} ~ {X_train_flat.std(axis=0).max():.4f}")

print("\nX_val 검증")
print(f"  mean 범위: {X_val_flat.mean(axis=0).min():.3f} ~ {X_val_flat.mean(axis=0).max():.3f}")
print(f"  std 범위:  {X_val_flat.std(axis=0).min():.3f} ~ {X_val_flat.std(axis=0).max():.3f}")

print("\nX_test 검증 (c6은 다른 cutter라 어느 정도 차이는 정상)")
print(f"  mean 범위: {X_test_flat.mean(axis=0).min():.3f} ~ {X_test_flat.mean(axis=0).max():.3f}")
print(f"  std 범위:  {X_test_flat.std(axis=0).min():.3f} ~ {X_test_flat.std(axis=0).max():.3f}")

for name, arr in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    print(f"{name}: NaN={np.isnan(arr).sum()}, Inf={np.isinf(arr).sum()}")

# ===== 18. 최종 산출물 저장 =====
OUTPUT_DIR = Path("preprocessed")
OUTPUT_DIR.mkdir(exist_ok=True)

np.save(OUTPUT_DIR / "X_train.npy", X_train)
np.save(OUTPUT_DIR / "y_train.npy", y_train)
np.save(OUTPUT_DIR / "X_val.npy", X_val)
np.save(OUTPUT_DIR / "y_val.npy", y_val)
np.save(OUTPUT_DIR / "X_test.npy", X_test)
np.save(OUTPUT_DIR / "y_test.npy", y_test)

with open(OUTPUT_DIR / "feature_columns.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(feat_cols))

print(f"\n저장 완료: {OUTPUT_DIR}/ 폴더에 X_train.npy, y_train.npy, X_val.npy, y_val.npy, X_test.npy, y_test.npy, feature_columns.txt")